In [ ]:
# AI 신약개발 파이프라인 실험 일지

**날짜:** 2025년 10월 19일
**목표:** `Pocket2Mol`, `Binding Affinity`, `ADMET` 모델을 `drugai`와 `Pocket2Mol` 두 개의 Conda 환경에서 Jupyter Lab으로 자동화.

---

## 1. 환경 설정: Jupyter Lab 실행 오류

* **문제:** `(drugai)` 환경에서 `jupyter lab` 실행 시, `Jupyter command 'jupyter-lab' not found` 오류 발생.
* **원인:** `drugai` 환경에 `jupyter`의 핵심(core)은 설치되어 있으나, `jupyterlab` 패키지 자체는 미설치 상태였음.
* **해결:** `(drugai)` 환경에 `conda install -c conda-forge jupyterlab` 명령어로 `jupyterlab` 설치.

---

## 2. 환경 연동: `A_Environment_Check.ipynb` 디버깅

* **문제 1:** `conda run -n pocket2mol ...` 실행 시, `EnvironmentLocationNotFound` 오류 발생.
* **원인 1:** 가상환경 이름이 `pocket2mol` (소문자)이 아니었음.
* **해결 1:** `!conda env list` 실행. 실제 환경 이름이 **`Pocket2Mol`** (대소문자 구분)임을 확인. 코드의 `POCKET2MOL_ENV_NAME` 변수 수정.

* **문제 2:** `스크립트 파일을 찾을 수 없습니다!` (`.../ipynb/pocket2mol_tree/scripts/sample_for_pdb.py`)
* **원인 2:** `pocket2mol_tree`라는 폴더가 아니라, `/home/nudge/Pocket2Mol`이 실제 프로젝트 폴더였으며 `scripts` 하위 폴더도 없었음.
* **해결 2:** `POCKET2MOL_DIR` 변수를 `"/home/nudge/Pocket2Mol"`로 수정하고, `script_path`에서 "scripts/" 경로 제거.

* **결과:** `A_Environment_Check.ipynb`가 `sample_for_pdb.py --help`를 성공적으로 호출. **두 환경 간의 연동 확인 완료.**

---

## 3. 파이프라인 [셀 2] (Pocket2Mol) 디버깅

* **문제 1:** `sample_for_pdb.py: error: unrecognized arguments: 15.5 -5.0 ...`
* **원인 1:** `sample_for_pdb.py` 스크립트가 `--center`와 `--bbox_size` 인자를 특이한 형식으로 받았음.
* **해결 1:**
    * `--center`는 쉼표로 구분된 단일 문자열 (예: `"0.0,0.0,-25.721"`)로 전달하도록 `center_str` 수정.
    * `--bbox_size`는 공백 구분 숫자 3개가 아닌 **단일 숫자** (예: `20.0`)로 전달하도록 `[셀 1]`의 `POCKET_BBOX_SIZE` 변수 자체를 `float`으로 변경하고, `[셀 2]`의 `bbox_str` 생성 코드를 `str(POCKET_BBOX_SIZE)`로 수정.

* **문제 2:** `TypeError: 'float' object is not iterable`
* **원인 2:** `[셀 1]`에서 `POCKET_BBOX_SIZE`를 `float`으로 수정한 뒤, `[셀 2]`의 `bbox_str` 생성 코드(`map(str, ...)` )를 수정하지 않고 실행함.
* **해결 2:** `bbox_str` 생성 코드를 `str(POCKET_BBOX_SIZE)`로 최종 수정.

* **문제 3:** `NameError: name 'POCKET2MOL_ENV_NAME' is not defined`
* **원인 3:** `[셀 1]` (설정 셀)을 실행하지 않고 `[셀 2]`를 바로 실행하여 변수가 메모리에 로드되지 않음.
* **해결 3:** **Jupyter 노트북은 항상 [셀 1]부터 순서대로 실행해야 함**을 확인.

* **문제 4:** `Pocket2Mol` 모델은 100% 성공적으로 실행되었으나 (`Finished 100+`), `❌ SMILES.txt 파일이 생성되지 않았습니다.` 오류 발생.
* **원인 4:** `sample_for_pdb.py` 스크립트가 `--outdir`로 지정된 폴더(`.../1_generated_molecules/`) **안에** `sample_for_pdb_...` 라는 **동적 이름의 하위 폴더**를 자동으로 만들고, 그 **안에** `SMILES.txt`를 저장함.
* **해결 4:** `[셀 2]`의 파일 확인 로직을 `os.path.exists` (고정 경로) 대신, `glob.glob("**/SMILES.txt", recursive=True)` (재귀 검색)으로 수정. `global smiles_file_path`를 사용해 찾은 경로를 다음 셀들로 전달.

---

## 4. 파이프라인 [셀 3] (Binding Affinity) 디버깅

* **문제:** `❌ Error: Model or Protein file not found.`
* **원인:** `[셀 1]`의 `AFFINITY_MODEL_CKPT` 변수 경로가 `/path/to/...` 플레이스홀더였음. `!ls -l {AFFINITY_MODEL_CKPT}` 디버깅 셀로 `No such file or directory` 확인.
* **해결:** `[셀 1]`의 `AFFINITY_MODEL_CKPT` 변수에 실제 `.pt` 파일의 절대 경로를 입력하여 수정.

---

## 5. 파이프라인 [셀 4] (ADMET) 디버깅

* **문제:** `ModuleNotFoundError: No module named 'src.admet'`
* **원인:** 사용자가 제공한 실제 실행 명령어를 분석한 결과,
    1.  모듈 이름이 `src.admet...`가 아니라 **`src.admet_multi.predict_admet_gnn`**이었음.
    2.  체크포인트 폴더 경로(`ADMET_MODEL_CKPT_DIR`)가 잘못되었음. (실제: `.../drugai/models`)
    3.  `--model_type custom` 등 **6개의 필수 실행 인자**가 누락되었음.
* **해결:**
    1.  `[셀 1]`의 `ADMET_MODEL_CKPT_DIR` 변수 경로를 실제 `.../drugai/models` 폴더로 수정.
    2.  `[셀 4]`의 `cmd3` 명령어를 `PYTHONPATH={DRUGAI_DIR}`로 시작하도록 하고, 올바른 모듈 이름(`src.admet_multi...`)과 6개의 필수 인자를 모두 추가하여 수정.

---

## 6. 최종 성공

* **결과:** `[셀 1]`부터 `[셀 5]`까지 모든 수정사항을 반영하여 재실행, `[셀 5]`에서 최종 병합 테이블 출력 **성공**.
* **다음 단계:** `C_Analyze_Results.ipynb` 노트북을 새로 생성하여 `FINAL_PIPELINE_RESULTS.csv` 파일을 로드하고, `pandas`로 필터링 및 분석 시작.

In [ ]:
# AI 신약개발 파이프라인 구축 오답노트

**프로젝트 목표:** `Pocket2Mol`, `Binding Affinity`, `ADMET` 모델을 연동하여 자동화된 파이프라인 구축 (`drugai`, `Pocket2Mol` Conda 환경 분리 사용)

**작성일:** 2025년 10월 21일

---

## 1. 환경 설정 문제

### 오류 내용
`(drugai)` 환경 터미널에서 `jupyter lab` 실행 시 `Jupyter command 'jupyter-lab' not found` 오류 발생.

### 원인 분석
`drugai` 환경에는 `jupyter` 코어만 설치되어 있고, `jupyterlab` 인터페이스 패키지는 설치되지 않았음.

### 해결 과정
`drugai` 환경에 `mamba install -c conda-forge jupyterlab -y` (또는 `conda install`) 명령어로 `jupyterlab` 설치 후 정상 실행 확인. ✅

---

## 2. Conda 환경 연동 문제 (`A_Environment_Check.ipynb`)

### 오류 내용 1
`conda run -n pocket2mol ...` 실행 시 `EnvironmentLocationNotFound` 발생.

### 원인 분석 1
Conda 환경 이름은 대소문자를 구분함. 실제 환경 이름은 `pocket2mol`이 아니라 `Pocket2Mol`이었음.

### 해결 과정 1
노트북에서 `!conda env list` 명령 실행하여 정확한 환경 이름(`Pocket2Mol`) 확인 후, 코드 내 `POCKET2MOL_ENV_NAME` 변수 수정. ✅

### 오류 내용 2
`FileNotFoundError: .../ipynb/pocket2mol_tree/scripts/sample_for_pdb.py` 발생 (스크립트 경로 오류).

### 원인 분석 2
-   `pocket2mol_tree` 폴더가 아니라 `/home/nudge/Pocket2Mol`이 실제 프로젝트 폴더였음.
-   `scripts` 하위 폴더가 존재하지 않았음.

### 해결 과정 2
-   노트북 코드의 `POCKET2MOL_DIR` 변수를 정확한 절대 경로(`/home/nudge/Pocket2Mol`)로 수정.
-   `script_path` 생성 시 불필요한 `/scripts/` 경로 제거. ✅

---

## 3. Pocket2Mol 실행 오류 (`B_Run_Pipeline.ipynb` - [셀 2])

### 오류 내용 1
`sample_for_pdb.py: error: unrecognized arguments: 15.5 -5.0 ...` (`--center` 인자 문제).

### 원인 분석 1
`sample_for_pdb.py` 스크립트가 `--center` 인자를 공백 구분 숫자 3개가 아닌, **쉼표로 구분된 문자열 1개** (예: `"x,y,z"`) 형식으로 기대함.

### 해결 과정 1
`center_str` 변수 생성 코드를 `f"\"{','.join(map(str, POCKET_CENTER))}\""` 로 수정하여 올바른 형식으로 전달. ✅

### 오류 내용 2
`sample_for_pdb.py: error: argument --bbox_size: invalid float value: '20.0,20.0,20.0'`.

### 원인 분석 2
`--bbox_size` 인자는 쉼표 구분 문자열이 아니라 **단일 숫자(float)**를 기대함.

### 해결 과정 2
-   `[셀 1]`의 `POCKET_BBOX_SIZE` 변수 타입을 리스트에서 단일 `float` (예: `20.0`)으로 변경.
-   `[셀 2]`의 `bbox_str` 생성 코드를 `str(POCKET_BBOX_SIZE)`로 수정. ✅

### 오류 내용 3
`TypeError: 'float' object is not iterable` (`bbox_str` 생성 중).

### 원인 분석 3
`POCKET_BBOX_SIZE`를 `float`으로 변경 후, `bbox_str` 생성 코드(`map(str, ...)`)를 수정하지 않고 실행함.

### 해결 과정 3
`bbox_str` 생성 코드를 `str(POCKET_BBOX_SIZE)`로 최종 수정. ✅

### 오류 내용 4
`NameError: name 'POCKET2MOL_ENV_NAME' is not defined`.

### 원인 분석 4
`[셀 1]` (설정 셀)을 실행하지 않고 `[셀 2]`를 먼저 실행하여 변수가 메모리에 로드되지 않음.

### 해결 과정 4
Jupyter 노트북 셀은 **반드시 `[셀 1]`부터 순서대로 실행**해야 함을 인지. ✅

### 오류 내용 5
Pocket2Mol 스크립트 로그는 성공 (`Finished 100+`)했으나, 노트북 결과는 `FileNotFoundError: ...SMILES.txt`.

### 원인 분석 5
-   `sample_for_pdb.py`가 `--outdir` 내부에 타임스탬프 기반의 **동적 하위 폴더**(`sample_for_pdb_...`)를 생성하고 그 안에 `SMILES.txt`를 저장함.
-   노트북의 파일 존재 여부 확인(`os.path.exists`)이 파일 시스템 쓰기 지연보다 빨라 발생하는 경합 상태(Race Condition) 가능성.

### 해결 과정 5
-   `[셀 2]` 코드 수정:
    -   `glob.glob("**/SMILES.txt", recursive=True)`를 사용하여 하위 폴더까지 동적으로 `SMILES.txt` 검색.
    -   `time.sleep(3)` 추가하여 파일 쓰기 시간 확보.
    -   `global smiles_file_path` 선언하여 찾은 파일 경로를 이후 셀들에서 사용하도록 함. ✅

### 오류 내용 6
`ValueError: probabilities do not sum to 1` (`sample_for_pdb.py` 내부 NumPy 오류).

### 원인 분석 6
`Pocket2Mol` 모델의 수치적 불안정성 문제로 추정됨. 특정 PDB(`3TCT`) 및 포켓 좌표 입력 시 발생.

### 해결 과정 6
`[셀 1]`의 `POCKET_CENTER` 또는 `POCKET_BBOX_SIZE` 값을 약간 변경하여 재시도. (사용자가 좌표 변경 후 오류 해결됨). ✅

---

## 4. Binding Affinity 실행 오류 (`B_Run_Pipeline.ipynb` - [셀 3])

### 오류 내용
`Error: Model or Protein file not found` (`screen.py` 실행 시).

### 원인 분석
`[셀 1]`에 설정된 `AFFINITY_MODEL_CKPT` 경로가 잘못되었음 (플레이스홀더 또는 존재하지 않는 파일). 노트북에 임시 디버깅 셀(`!ls -l {AFFINITY_MODEL_CKPT}`)을 추가하여 확인.

### 해결 과정
`[셀 1]`의 `AFFINITY_MODEL_CKPT` 변수에 실제 모델 파일(`.../checkpoints/egnn_best.pt`)의 **정확한 절대 경로** 입력. ✅

---

## 5. ADMET 실행 오류 (`B_Run_Pipeline.ipynb` - [셀 4])

### 오류 내용
`ModuleNotFoundError: No module named 'src.admet'`.

### 원인 분석
-   실제 모듈 이름이 `src.admet_multi.predict_admet_gnn` 이었음 (초기 추측 오류).
-   `ADMET_MODEL_CKPT_DIR` 경로가 잘못 설정됨 (실제: `.../drugai/models`).
-   `predict_admet_gnn.py` 스크립트 실행에 필요한 `--model_type custom` 등 **6개의 추가 인자 누락**.
-   `cd` 명령어로 작업 디렉토리 변경 시 Python 모듈 검색 경로(`PYTHONPATH`) 문제가 발생할 수 있음.

### 해결 과정
-   `[셀 1]`의 `ADMET_MODEL_CKPT_DIR` 경로를 실제 경로(`.../drugai/models`)로 수정.
-   `[셀 4]` 코드 수정:
    -   `module_path`를 `src.admet_multi.predict_admet_gnn`으로 변경.
    -   `cmd3` 명령어 시작 부분에 `PYTHONPATH={DRUGAI_DIR}` 추가 (안정성 확보).
    -   누락된 6개의 필수 인자 (`--model_type` 등) 모두 추가. ✅

---

## 6. 분석 노트북 (`C_Analyze_Results.ipynb`) 문제

### 오류 내용 1
`ModuleNotFoundError: No module named 'seaborn'`.

### 원인 분석 1
시각화 라이브러리 `seaborn`, `matplotlib` 미설치 (`drugai` 환경).

### 해결 과정 1
노트북에 `[셀 2.5]` 추가하여 `!mamba install -c conda-forge seaborn matplotlib -y` 실행하여 설치. ✅

### 오류 내용 2
`ArgumentError: ... legendFontSize=12 ... did not match C++ signature` (SMILES 이미지 생성 시).

### 원인 분석 2
`Draw.MolsToGridImage` 함수에 전달된 `legendFontSize` 인자가 특정 RDKit 백엔드(Cairo)와 호환되지 않음.

### 해결 과정 2
`[셀 4]` 코드에서 `Draw.MolsToGridImage` 호출 시 `legendFontSize=12` 인자 제거. ✅

### 불편함
`B` 노트북 실행 시마다 `C` 노트북 `[셀 1]`의 `CSV_PATH` (결과 파일 경로)를 수동으로 수정해야 함.

### 해결 과정
`C` 노트북 `[셀 1]` 코드 수정: `glob.glob("./pipeline_results_*")` 와 `max(..., key=os.path.getmtime)`을 사용하여 **가장 최신 결과 폴더를 자동으로 찾아** `CSV_PATH` 설정. ✅

---

## 7. 실제 리간드 테스트 (`E_Test_Known_Ligand.ipynb`) 문제

### 오류 내용 1
PDB `3TCT`의 실제 리간드를 `DIF` (디플루니살)로 잘못 인식.

### 원인 분석 1
나의 정보 오류.

### 해결 과정 1
사용자가 PDB 사이트에서 직접 확인한 올바른 리간드 ID **`3MI`**로 `[셀 1]`의 `LIGAND_ID` 변수 수정. ✅

### 오류 내용 2
`AttributeError: ... has no attribute 'rdVersion'` 또는 `__version__`.

### 원인 분석 2
설치된 RDKit 버전에 따라 버전 확인 속성 이름이 다름.

### 해결 과정 2
`[셀 1]` 코드 수정: `try-except` 블록으로 `Chem.__version__`, `rdBase._version`, `Chem.rdVersion` 순서로 로드 시도하여 호환성 확보. ✅

### 오류 내용 3
`/bin/sh: 1: Syntax error: "(" unexpected` (`predict.py` 실행 시).

### 원인 분석 3
SMILES 문자열에 포함된 괄호 `()`가 쉘 특수 문자로 해석되어 오류 발생.

### 해결 과정 3
`[셀 3]` 코드 수정: `cmd` 명령어 생성 시 `{ligand_smiles}` 변수를 **작은따옴표(`' '`)**로 감싸서 쉘 이스케이프 처리. ✅

---